<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/5_transformer_90_model/5_1_k_windows.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 5_1_k_windows


## Introducción y Resumen

El objetivo de esta notebook es generar las ventanas de entrada (X) y targets (y) para los horizontes de 30, 60 y 90 minutos, aplica el escalado de features sobre cada conjunto (train, valid, test) y finalmente guarda las ventanas escaladas en disco, dejándolas listas para el entrenamiento de modelos.

0. Configuración del Entorno

    Se conecta Google Drive y se clona el repositorio de trabajo. Se instalan e importan librerías necesarias como pandas, numpy, matplotlib y seaborn. Se cargan los datasets procesados previamente y se muestra un resumen de la información de los datasets.

1. Carga de datos

    Se importan los datasets procesados `mnq_train`, `mnq_valid` y `mnq_test`. Se revisa su estructura (filas, columnas, tipos de datos) y también de importa el listado de features seleccionados para cada ventana de tiempo.

2. Generación de ventanas X e y para train, valid y test

    En este paso se generan las ventanas de entrada (X) y los targets (y) para los conjuntos de entrenamiento, validación y prueba.
    Se trabaja con un window_size de 90 minutos y se construyen datasets independientes para cada horizonte de predicción: 30, 60 y 90 minutos.

3. Escalado de ventanas

    En este paso se aplica un proceso de normalización/estandarización a las ventanas generadas, utilizando un scaler entrenado únicamente con el set de entrenamiento para cada horizonte de predicción.
    De esta forma, se aseguran valores comparables entre features y se evita data leakage.
    El scaler ajustado se guarda para poder transformar consistentemente los conjuntos de validación y prueba.

4. Guardado de ventanas escaladas

    En este paso se almacenan en disco las ventanas ya escaladas de train, valid y test, correspondientes a cada horizonte de predicción (30, 60 y 90 minutos).
    Esto permite reutilizar los datasets en etapas posteriores sin necesidad de repetir el preprocesamiento.



## 0. Configuración del Entorno


### 0.1. Clonado de repositorio / Acceso a Drive

In [1]:
#Clonamos el repo
#LINK DE REPOSITORIO: https://github.com/GUNAPILLCO/neural_profit
#!git clone https://github.com/GUNAPILLCO/neural_profit.git

In [2]:
from google.colab import drive
drive.mount('/content/drive')
drive_path = "/content/drive/MyDrive/neural_profit"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### 0.2. Instalación de librerías


In [3]:
#!{sys.executable} -m pip install -q ta
#print("Librería instalada: technical-analysis")

### 0.3. Importación de librerías


In [4]:
import sys
import re
#Instalación de librería pandas_market_calendars
#!{sys.executable} -m pip install -q pandas_market_calendars
#print("Librería instalada: pandas_market_calendars")


from functools import reduce
# Utilidades generales
from datetime import datetime, timedelta
import os
import glob
import requests
import warnings
warnings.filterwarnings('ignore')

# Manejo y procesamiento de datos
#import ta
import pandas as pd
import numpy as np
from tabulate import tabulate
import matplotlib.pyplot as plt
# Calendario de mercados
#import pandas_market_calendars as mcal

#from ta.momentum import StochasticOscillator, ROCIndicator
#from ta.volatility import BollingerBands, AverageTrueRange

from scipy.stats import spearmanr
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import joblib

## 1. Carga de datos

### 1.1. Carga de datasets `mnq_train`, `mnq_valid` y `mnq_test`






In [5]:
def load_data(fold: str, data: str):

    data_path = f'{drive_path}/5_transformer_90_model/5_0_k_folds/fold_{fold}/{data}_{fold}.parquet'
    # Leer el archivo Parquet y cargarlo en un DataFrame
    df = pd.read_parquet(data_path)

    # Asegurar orden cronológico por índice
    df = df.sort_index()

    # Asegurar que el índice esté en formato datetime
    df.index = pd.to_datetime(df.index)

    # Crear una nueva columna 'date' con la fecha extraída del índice
    df['date'] = df.index.date

    # Reordenar columnas: 'date', 'time_str', y luego el resto
    cols = ['date'] + [col for col in df.columns if col not in ['date']]

    df = df[cols]

    return df

In [6]:
k_folds = [1, 2, 3, 4, 5]

In [7]:
mnq_train = {}
mnq_valid = {}
mnq_test  = {}

for k in k_folds:
    print(f"Cargando datos de Fold {k}..")

    mnq_train[k] = load_data(str(k), "train")
    mnq_valid[k] = load_data(str(k), "valid")
    mnq_test[k]  = load_data(str(k), "test")

Cargando datos de Fold 1..
Cargando datos de Fold 2..
Cargando datos de Fold 3..
Cargando datos de Fold 4..
Cargando datos de Fold 5..


### 1.2. Información de datasets


In [8]:
def info_dataset(df, name: str):

  # Contar valores únicos en la columna 'date'
  num_dias = df['date'].nunique()
  #print(f"\t{name}:\t{num_dias} días")

  # Filtrar valores válidos
  validos_por_dia = df.dropna(subset=['close']).groupby('date').size()

  # Calcular el promedio
  promedio_por_fecha = validos_por_dia.mean()
  #print(f"\tRegistros por día: {int(promedio_por_fecha)}")

  print(f"\t{name}:\t{num_dias} días con {int(promedio_por_fecha)} registros")

  primer_hora = df.index[0].strftime('%H:%M')
  ultima_hora = df.index[-1].strftime('%H:%M')
  zona_horaria = df.index[0].tzinfo

  #print(f"\tHora diaria de inicio {primer_hora}")
  #print(f"\tHora diaria de final {ultima_hora}")
  #print(f"\tZona horaria: {zona_horaria}\n")

  return num_dias, promedio_por_fecha

In [9]:
for k in k_folds:
    print(f"Fold {k}:")

    info_dataset(mnq_train[k], f"mnq_train_{k}")
    info_dataset(mnq_valid[k], f"mnq_valid_{k}")
    info_dataset(mnq_test[k],  f"mnq_test_{k}")

    print("\n")

Fold 1:
	mnq_train_1:	589 días con 301 registros
	mnq_valid_1:	118 días con 301 registros
	mnq_test_1:	132 días con 301 registros


Fold 2:
	mnq_train_2:	707 días con 301 registros
	mnq_valid_2:	118 días con 301 registros
	mnq_test_2:	132 días con 301 registros


Fold 3:
	mnq_train_3:	825 días con 301 registros
	mnq_valid_3:	118 días con 301 registros
	mnq_test_3:	132 días con 301 registros


Fold 4:
	mnq_train_4:	943 días con 301 registros
	mnq_valid_4:	118 días con 301 registros
	mnq_test_4:	132 días con 301 registros


Fold 5:
	mnq_train_5:	1061 días con 301 registros
	mnq_valid_5:	118 días con 301 registros
	mnq_test_5:	132 días con 301 registros




## 2. Generación de ventanas X e y para train, valid y test

Definimos el target de cada horizonte:

In [10]:
target_col_90 = "target_return_90"

Luego definimos el listado de features para cada horizonte:

In [11]:
features_90 = features_90 = [
    col for col in mnq_train[1].columns
    if col not in ["date", "target_return_90"]
]

In [12]:
features_90

['open',
 'high',
 'low',
 'close',
 'volume',
 'ire_60',
 'rev_mom_z_90',
 'roc_60',
 'bb_60',
 'momentum_5',
 'roc_20',
 'rev_mom_vol_z_60']

El `window_size` está condicionado por el feature que más historial necesita, en nuestro caso `roc_90`, `rev_mom_z_90` y `ire_90` necesitan 90 minutos previos para poder calcular su primer valor válido.

Si hacemos más corto el `window_size` corremos el riesgo de perder información o generar NaNs.

Y un `window_size` más largo?  por ahora experimentemos con 90.


In [13]:
window_size = 90

Definimos las rutas para las ventanas:

In [14]:
# Ruta base donde guardarás los folds
ruta_k_windows = f'{drive_path}/5_transformer_90_model/5_1_k_windows'
os.makedirs(ruta_k_windows, exist_ok=True)

In [15]:
def xy_paths_for_fold(k: int):
    # Crear carpeta del fold
    fold_path = os.path.join(ruta_k_windows, f"fold_{k}")
    os.makedirs(fold_path, exist_ok=True)
    base = f'{drive_path}/5_transformer_90_model/5_1_k_windows/fold_{k}'
    return {
        "X_train": f"{base}/X_train_{k}.npz",
        "y_train": f"{base}/y_train_{k}.npz",
        "X_valid": f"{base}/X_valid_{k}.npz",
        "y_valid": f"{base}/y_valid_{k}.npz",
        "X_test":  f"{base}/X_test_{k}.npz",
        "y_test":  f"{base}/y_test_{k}.npz",
    }

In [16]:
K = 5

rutas_ventanas = {
    k: xy_paths_for_fold(k)
    for k in range(1, K + 1)
}

In [17]:
rutas_ventanas

{1: {'X_train': '/content/drive/MyDrive/neural_profit/5_transformer_90_model/5_1_k_windows/fold_1/X_train_1.npz',
  'y_train': '/content/drive/MyDrive/neural_profit/5_transformer_90_model/5_1_k_windows/fold_1/y_train_1.npz',
  'X_valid': '/content/drive/MyDrive/neural_profit/5_transformer_90_model/5_1_k_windows/fold_1/X_valid_1.npz',
  'y_valid': '/content/drive/MyDrive/neural_profit/5_transformer_90_model/5_1_k_windows/fold_1/y_valid_1.npz',
  'X_test': '/content/drive/MyDrive/neural_profit/5_transformer_90_model/5_1_k_windows/fold_1/X_test_1.npz',
  'y_test': '/content/drive/MyDrive/neural_profit/5_transformer_90_model/5_1_k_windows/fold_1/y_test_1.npz'},
 2: {'X_train': '/content/drive/MyDrive/neural_profit/5_transformer_90_model/5_1_k_windows/fold_2/X_train_2.npz',
  'y_train': '/content/drive/MyDrive/neural_profit/5_transformer_90_model/5_1_k_windows/fold_2/y_train_2.npz',
  'X_valid': '/content/drive/MyDrive/neural_profit/5_transformer_90_model/5_1_k_windows/fold_2/X_valid_2.npz'

### 2.0. Funciones

#### Función para generar ventanas

Genera ventana consecutivas y no aleatorias, es decir: ventanas deslizantes (sliding windows) dentro de cada día.

- Agrupamiento diario: Cada iteración toma un día completo del dataset (917 días en total). Dentro de ese grupo tenemos 301 registros minuto a minuto.

- Iteración dentro del día: genera una ventana que empieza en el minuto i  termina en i + windows_size-1.

Ejemplo si window_size = 90 y tenemos 301 minutos:

  | Iteración | Ventana usada     | Target extraído       |
  | --------- | ----------------- | --------------------- |
  | i = 0     | registros 0–89    | target = registro 89  |
  | i = 1     | registros 1–90    | target = registro 90  |
  | i = 2     | registros 2–91    | target = registro 91  |
  | ...       | ...               | ...                   |
  | i = 210   | registros 210–299 | target = registro 299 |

Esto da 301 - 90 = 211 ventanas por día, todas consecutivas.


In [18]:
#Función para generar ventanas y vectorizarlas
def generar_ventanas(df, features, target_col, window_size):
    X, y = [], []

    #1. Agrupamiento diario: Cada iteración toma un día completo del dataset (917 días en total). Dentro de ese grupo tenemos 301 registros minuto a minuto.
    for fecha, grupo in tqdm(df.groupby("date"), desc="Procesando días"):
        grupo = grupo.reset_index(drop=True)

        #2. Iteración dentro del día: genera una ventana que empieza en el minuto i  termina en i + windows_size-1.
        for i in range(len(grupo) - window_size):
            ventana = grupo.loc[i:i+window_size-1, features]
            if ventana.isnull().any().any():
                continue

            # 3. Toma las columnas listadas en features (por ejemplo, 10 features por minuto) y las aplanas en un solo vector 1D de longitud window_size × len(features) (= 900 si window_size=90 y len(features)=10).
            vector = ventana.values.flatten()

            #4. El target de la ventana es el valor del registro en el último minuto de la ventana, o sea el del minuto i + window_size - 1.
            #(No es “futuro”, sino el último dentro de la ventana).
            target = grupo.loc[i+window_size-1, target_col]

            X.append(vector)
            y.append(target)

      # Se obtiene:
      # X.shape = (n_ventanas_totales, window_size * n_features)
      # y.shape = (n_ventanas_totales,)
    return np.array(X), np.array(y)

Cada ventana contiene:

- 90 minutos consecutivos de datos de un mismo día.
- En cada minuto, 10 o 12 features (por ejemplo: open, high, low, close, volume, etc.).
- Esos 90×10 (ó 12) valores se aplanan en un vector de 900 (ó 1080) elementos.
- El target asociado es el valor del minuto siguiente al final de la ventana (o del último minuto, según definas).
- Por día se generan 301 − 90 = 211 ventanas, todas superpuestas y consecutivas.
- Repetido en los 917 días de train,  se obtiene 193 487 ventanas en total.



#### Función para generar xy de acuerdo a horizonte de tiempo

In [19]:
def generar_xy (
    df_train,
    df_valid,
    df_test,
    features,
    target: str,
    window_size: int,
    path_xy_train : str,
    path_xy_valid : str,
    path_xy_test : str
    ):

  if not os.path.exists(path_xy_train):
      print(f'El archivo no existe -> Generando X_train e y_train para {target}: ')
      X_train, y_train = generar_ventanas(df_train, features, target, window_size)
      np.savez_compressed(path_xy_train, X=X_train, y=y_train)
      print("Guardado:", path_xy_train)
  else:
      print("Ya existe -> Cargando desde disco:", path_xy_train)
      data_train = np.load(path_xy_train)
      X_train, y_train = data_train["X"], data_train["y"]

  if not os.path.exists(path_xy_valid):
      print('El archivo no existe -> Generando X_valid e y_valid: ')
      X_valid, y_valid = generar_ventanas(df_valid, features, target, window_size)
      np.savez_compressed(path_xy_valid, X=X_valid, y=y_valid)
      print("Guardado:", path_xy_valid)
  else:
      print("Ya existe -> Cargando desde disco:", path_xy_valid)
      data_valid = np.load(path_xy_valid)
      X_valid, y_valid = data_valid["X"], data_valid["y"]

  if not os.path.exists(path_xy_test):
      print('El archivo no existe -> Generando X_test e y_test: ')
      X_test, y_test = generar_ventanas(df_test, features, target, window_size)
      np.savez_compressed(path_xy_test, X=X_test, y=y_test)
      print("Guardado:", path_xy_test)
  else:
      print("Ya existe -> Cargando desde disco:", path_xy_test)
      data_test = np.load(path_xy_test)
      X_test, y_test = data_test["X"], data_test["y"]

  return X_train, y_train, X_valid, y_valid, X_test, y_test

#### Función para revisar información de ventanas

In [20]:
def xy_info(fold: str, X_train, y_train, X_valid, y_valid, X_test, y_test):
    print(f'Información de Fold {fold} :')

    for name, X, y in [
        ("entrenamiento", X_train, y_train),
        ("validación", X_valid, y_valid),
        ("testeo", X_test, y_test),
    ]:
        print(f'\nSet de {name}:')
        print(f'\t{X.shape[0]} ventanas (n_samples).')

        if X.ndim == 2:

            print(f'\t{X.shape[1]} features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_{fold})')
        elif X.ndim == 3:

            print(f'\t{X.shape[1]} pasos en lookback × {X.shape[2]} features por paso. Dimensión 3D: (n_samples, window_size, len(features_{fold})')

        print(f'\t{y.shape[0]} targets.')
        #print(f'\tDistribución y: mean={y.mean():.6f}, std={y.std():.6f}, min={y.min():.6f}, max={y.max():.6f}')

### 2.1. Generación de ventanas por fold

In [21]:
import gc

# Diccionarios para las ventanas
X_train_dict = {}
y_train_dict = {}
X_valid_dict = {}
y_valid_dict = {}
X_test_dict  = {}
y_test_dict  = {}

for k in k_folds:
    print(f'Fold {k}:')

    X_train, y_train, X_valid, y_valid, X_test, y_test = generar_xy(
        mnq_train[k],
        mnq_valid[k],
        mnq_test[k],
        features_90,
        target_col_90,
        window_size,
        rutas_ventanas[k]['X_train'],
        rutas_ventanas[k]['X_valid'],
        rutas_ventanas[k]['X_test']
    )

    # Guardar en diccionarios
    X_train_dict[k] = X_train
    y_train_dict[k] = y_train
    X_valid_dict[k] = X_valid
    y_valid_dict[k] = y_valid
    X_test_dict[k]  = X_test
    y_test_dict[k]  = y_test

    print(f"  - Ventanas almacenadas en diccionarios para el fold {k}")
    print("-" * 40)

Fold 1:
Ya existe -> Cargando desde disco: /content/drive/MyDrive/neural_profit/5_transformer_90_model/5_1_k_windows/fold_1/X_train_1.npz
Ya existe -> Cargando desde disco: /content/drive/MyDrive/neural_profit/5_transformer_90_model/5_1_k_windows/fold_1/X_valid_1.npz
Ya existe -> Cargando desde disco: /content/drive/MyDrive/neural_profit/5_transformer_90_model/5_1_k_windows/fold_1/X_test_1.npz
  - Ventanas almacenadas en diccionarios para el fold 1
----------------------------------------
Fold 2:
Ya existe -> Cargando desde disco: /content/drive/MyDrive/neural_profit/5_transformer_90_model/5_1_k_windows/fold_2/X_train_2.npz
Ya existe -> Cargando desde disco: /content/drive/MyDrive/neural_profit/5_transformer_90_model/5_1_k_windows/fold_2/X_valid_2.npz
Ya existe -> Cargando desde disco: /content/drive/MyDrive/neural_profit/5_transformer_90_model/5_1_k_windows/fold_2/X_test_2.npz
  - Ventanas almacenadas en diccionarios para el fold 2
----------------------------------------
Fold 3:
Ya e

In [22]:
#Veamos que tenemos guardado
[v for v in globals().keys() if v.startswith("X_train_")]

['X_train_dict']

In [23]:
for k in k_folds:
    xy_info(
        f"Fold {k}",
        X_train_dict[k],
        y_train_dict[k],
        X_valid_dict[k],
        y_valid_dict[k],
        X_test_dict[k],
        y_test_dict[k]
    )

Información de Fold Fold 1 :

Set de entrenamiento:
	124279 ventanas (n_samples).
	1080 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_Fold 1)
	124279 targets.

Set de validación:
	24898 ventanas (n_samples).
	1080 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_Fold 1)
	24898 targets.

Set de testeo:
	27852 ventanas (n_samples).
	1080 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_Fold 1)
	27852 targets.
Información de Fold Fold 2 :

Set de entrenamiento:
	149177 ventanas (n_samples).
	1080 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_Fold 2)
	149177 targets.

Set de validación:
	24898 ventanas (n_samples).
	1080 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_Fold 2)
	24898 targets.

Set de testeo:
	27852 ventanas (n_samples).
	1080 features totales por